# Xây dựng và đánh giá mô hình dự đoán mua hàng

Notebook này so sánh một quy tắc đơn giản dựa vào lần mua gần nhất với Logistic Regression và LightGBM dùng 18 biến hành vi. Sau đó xem kết quả mô hình thử nghiệm cuối cùng có thêm nhịp mua theo chu kỳ. Các thử nghiệm chi tiết được lưu riêng trong [khu vực nghiên cứu](research/).

Điều cần dự đoán là hộ có bất kỳ giỏ hàng nào trong 7 ngày sau mốc chốt. Cùng một hộ có thể xuất hiện ở nhiều mốc. Kết quả không đo tác động của phiếu giảm giá; tập kiểm tra cuối đã từng được xem trong quá trình phát triển.

## 1. Đọc dữ liệu huấn luyện và dữ liệu chọn mô hình

Kiểm tra đúng cấu trúc bảng và chỉ lấy các cột được phép dùng. Mã hộ, ngày chốt, cờ bất thường và nhãn cần dự đoán không phải biến đầu vào. Chưa đọc tập kiểm tra cuối ở bước này.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss,log_loss,roc_auc_score
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
if (ROOT/'.deps').is_dir():sys.path.insert(0,str(ROOT/'.deps'))
from propensity_v2.prepared_data import load_prepared_split
from lightgbm import LGBMClassifier
DATA=ROOT/'artifacts'/'data_preparation'
train,features=load_prepared_split(DATA,'train')
validation,vfeatures=load_prepared_split(DATA,'validation')
assert features==vfeatures and len(features)==18
assert train.snapshot_day.max()+7<validation.snapshot_day.min()
print('Train:',len(train),'| Validation:',len(validation),'| Feature:',len(features))

Train: 129834 | Validation: 27480 | Feature: 18


## 2. So với quy tắc đơn giản trước khi dùng mô hình phức tạp hơn

Mốc so sánh ban đầu lấy tỷ lệ mua của các nhóm theo số ngày từ lần mua gần nhất; nhóm không có dữ liệu dùng tỷ lệ chung của tập huấn luyện. Logistic Regression chỉ học cách chuẩn hóa từ tập huấn luyện; LightGBM dùng cấu hình khởi đầu. Hai mô hình cùng dùng 18 biến và dữ liệu đến mốc 537. Đây là bước so sánh ban đầu, chưa phải mô hình thử nghiệm cuối.

In [2]:
labels=['0–7','8–14','15–30','31–60','61–90','91+']
bins=[-1,7,14,30,60,90,np.inf]
train_group=pd.cut(train.recency_days,bins=bins,labels=labels)
val_group=pd.cut(validation.recency_days,bins=bins,labels=labels)
rates=train.assign(group=train_group).groupby('group',observed=False).target_any_basket.mean()
rates=rates.fillna(train.target_any_basket.mean())
probabilities={'Baseline recency':val_group.map(rates).astype(float).to_numpy()}
logistic=make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,random_state=42))
lightgbm=LGBMClassifier(n_estimators=200,learning_rate=.05,num_leaves=15,min_child_samples=100,random_state=42,verbosity=-1,n_jobs=2)
for name,model in [('Logistic Regression',logistic),('LightGBM',lightgbm)]:
    model.fit(train[features],train.target_any_basket)
    probabilities[name]=model.predict_proba(validation[features])[:,1]
assert all(np.isfinite(p).all() and ((p>=0)&(p<=1)).all() for p in probabilities.values())
print('Đã huấn luyện và dự báo trên validation.')

Đã huấn luyện và dự báo trên validation.


## 3. Đánh giá trên giai đoạn dùng để chọn mô hình

Brier và log loss đo chất lượng xác suất: càng thấp càng tốt. ROC-AUC cho biết khả năng xếp hộ mua cao hơn hộ không mua. Lift ở nhóm 10% xác suất cao nhất được tính riêng từng mốc rồi lấy trung bình; khi đồng điểm, không dùng nhãn thật để quyết định chọn ai. Lift cao không chứng minh việc liên hệ khách sẽ tạo thêm doanh số.

In [3]:
def weekly_lift(frame,p):
    values=[]
    y=frame.target_any_basket.to_numpy()
    for ix in frame.groupby('snapshot_day').indices.values():
        target=y[ix];q=p[ix];k=int(np.ceil(.1*len(ix)))
        threshold=np.partition(q,len(q)-k)[len(q)-k];higher=q>threshold;ties=q==threshold
        precision=(target[higher].sum()+(k-higher.sum())*target[ties].mean())/k
        values.append(precision/target.mean() if target.mean() else np.nan)
    return np.nanmean(values)
comparison=[]
for name,p in probabilities.items():
    comparison.append({'model':name,'brier':brier_score_loss(validation.target_any_basket,p),
                       'log_loss':log_loss(validation.target_any_basket,p,labels=[0,1]),
                       'roc_auc':roc_auc_score(validation.target_any_basket,p),
                       'weekly_lift_mean':weekly_lift(validation,p)})
comparison=pd.DataFrame(comparison).set_index('model');display(comparison)

,brier,log_loss,roc_auc,weekly_lift_mean
model,,,,
Baseline recency,0.199591,0.584073,0.726843,1.313018
Logistic Regression,0.166371,0.501671,0.833017,1.777743
LightGBM,0.164389,0.495405,0.836662,1.786582


## 4. Kiểm tra thêm liệu nhịp mua theo chu kỳ có ích không

Thử thêm khoảng cách điển hình và mức biến động giữa các ngày mua trong 180 ngày, cùng cờ cho biết lịch sử có đủ để tính hay không. Qua bốn lần kiểm tra theo thời gian trên tập huấn luyện, Brier và log loss đều giảm ở cả hai mô hình, nhưng lift không tăng đều. Kết quả này chưa bảo đảm sẽ tốt hơn trên dữ liệu tương lai.

Bảng dưới đọc kết quả đã lưu. Muốn chạy lại bốn lần kiểm tra, mở [notebook thí nghiệm theo thời gian](research/rolling_backtest_chu_ky_mua.ipynb). Ô này không huấn luyện lại mô hình.

In [4]:
path=ROOT/'artifacts'/'research'/'cadence_rolling_backtest'/'fold_feature_differences.csv'
if not path.exists():raise FileNotFoundError('Chạy research/rolling_backtest_chu_ky_mua.ipynb để tạo kết quả')
folds=pd.read_csv(path)
view=folds.loc[folds.train_scenario.eq('toàn bộ')&folds.eval_scenario.eq('toàn bộ'),
               ['fold','model','delta_brier','delta_log_loss','delta_roc_auc','delta_weekly_lift']]
display(view)

,fold,model,delta_brier,delta_log_loss,delta_roc_auc,delta_weekly_lift
0,1,Logistic Regression,-0.000913,-0.002397,0.001486,0.003847
2,1,LightGBM,-0.000814,-0.002120,0.001509,0.005124
8,2,Logistic Regression,-0.000631,-0.001968,0.000882,-0.000047
10,2,LightGBM,-0.000813,-0.002228,0.001659,0.006365
16,3,Logistic Regression,-0.000743,-0.002313,0.001114,0.000000
18,3,LightGBM,-0.000459,-0.001023,0.001021,0.001026
24,4,Logistic Regression,-0.000893,-0.003267,0.001429,0.003678
26,4,LightGBM,-0.000400,-0.000918,0.000748,-0.012535


## 5. Mô hình thử nghiệm cuối cùng và kết quả đã lưu

Phương án được chọn để minh họa là LightGBM với 18 biến ban đầu và các biến nhịp mua. Mô hình học đến ngày 439; phép hiệu chỉnh xác suất học ở các mốc 453–481 và được chọn ở các mốc 495–537. Theo quy tắc đặt trước, giữ xác suất gốc của mô hình. Bảng dưới đọc kết quả đã lưu từ [nghiên cứu hiệu chỉnh xác suất](research/calibration_va_danh_gia_poc.ipynb).

**Lưu ý khi so sánh:** mô hình ở mục 2 học đến ngày 537, còn phương án ở mục này chỉ học đến ngày 439. Không coi chúng là cùng một thí nghiệm. Kết quả trên tập kiểm tra cuối mang tính tham khảo vì tập này đã từng được xem.

In [5]:
poc_path=ROOT/'artifacts'/'calibration_final_poc'/'metrics.csv'
if not poc_path.exists():raise FileNotFoundError('Chạy research/calibration_va_danh_gia_poc.ipynb để tạo kết quả POC')
poc_metrics=pd.read_csv(poc_path)
display(poc_metrics)

,split,feature_set,rows,brier,log_loss,roc_auc,weekly_lift_mean
0,validation,18 feature,27480,0.165599,0.498827,0.834966,1.785994
1,validation,18 + chu kỳ,27480,0.165523,0.498580,0.835198,1.791388
2,test,18 feature,24994,0.168325,0.505472,0.829951,1.822417
3,test,18 + chu kỳ,24994,0.168401,0.505456,0.829773,1.818039


## 6. Kết luận và tạo danh sách xác suất

Các mô hình cho tín hiệu dự báo tốt hơn quy tắc chỉ dựa vào lần mua gần nhất. Biến chu kỳ cải thiện chất lượng xác suất nhưng lợi ích về lift nhỏ và chưa ổn định. Xác suất mua không cho biết mức tăng doanh số do chiến dịch hay lợi tức đầu tư. Bước tiếp theo là [tạo danh sách xác suất](deployment_gia_lap_propensity.ipynb); báo cáo cần nhắc lại rằng tập kiểm tra cuối đã xem và không có nhãn sau ngày 711.